# LSTM

In [ ]:
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple


## Load

In [ ]:
path = '../data/wyndham_smartbin_filllevel.json'

# Load
with open(path, 'r') as f:
    json_object = json.load(f)

## Isolate

In [ ]:
rows = []
for feature in json_object["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    rows.append({
        "timestamp": props["timestamp"],
        "serialnumber": props["serialNumber"],
        "latestfullness": props["latestFullness"]
    })

df = pd.DataFrame(rows)

## What we have so far

In [ ]:
display(df.head())

--

In [ ]:
#convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

#sort with serialnumber AND within that with timestamp
df = df.sort_values(["serialnumber", "timestamp"]).reset_index(drop=True)

df.head()


## Drop everything but first year

In [ ]:
# keep only the first year of data per serialnumber
df = df[
    df["timestamp"]
    <= df.groupby("serialnumber")["timestamp"].transform("min") + pd.DateOffset(years=1)
]

df.head()

## create "target" because we want to predict the next days fill-level. (not the current day)

In [ ]:
df["target"] = df.groupby("serialnumber")["latestfullness"].shift(-1)

# drop rows where next value doesn't exist (end of each bin series)
df = df.dropna(subset=["target"]).reset_index(drop=True)

df[["timestamp","serialnumber","latestfullness","target"]].head(10)
print("Shape:" ,df.shape)

## Convert to sequences

In [ ]:
SEQ_LEN = 30
TEST_FRAC = 0.2

X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []

for serial, g in df.groupby("serialnumber"):
    vals = g["latestfullness"].to_numpy(np.float32)
    y    = g["target"].to_numpy(np.float32) #the same but shifted by one

    # if len(g) <= SEQ_LEN:
    #     continue

    X_seq, y_seq = [], []
    for i in range(SEQ_LEN, len(g)):
        X_seq.append(vals[i-SEQ_LEN:i]) #the previous 14 samples
        y_seq.append(y[i]) #the current sample

    X_seq = np.array(X_seq, dtype=np.float32)[:, :, None]  # (samples, SEQ_LEN, 1)
    y_seq = np.array(y_seq, dtype=np.float32)

    split = int((1 - TEST_FRAC) * len(X_seq))
    X_train_list.append(X_seq[:split]); y_train_list.append(y_seq[:split])
    X_test_list.append(X_seq[split:]);  y_test_list.append(y_seq[split:])

X_train = np.concatenate(X_train_list, axis=0)
X_test  = np.concatenate(X_test_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
y_test  = np.concatenate(y_test_list, axis=0)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)


## Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_flat = X_train.reshape(-1, 1)
X_test_flat  = X_test.reshape(-1, 1)

X_train = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_test  = scaler.transform(X_test_flat).reshape(X_test.shape)


## Train and Predict LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

model = Sequential([
    LSTM(100, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(1)
])

model.compile(optimizer=Adam(), loss="mse")

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=70,
    validation_split=0.2,
    verbose=1
)

y_pred = model.predict(X_test).reshape(-1)


## Print metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


Hab gestopped beim Gedanken: was wenn ich mal nur das erste jahr nehm. So wie sies auch im PAper gemacht haben. Vl is dann RMSE kleiner auch.